# One-Sample t-Test Analysis: GNN (10-run distribution) vs. Random Forest (fixed value)

**Input** (8 files — the direct outputs of the multi-run cells appended to each of your
4 notebooks; each notebook's `summarize_runs()` call produces one `_overall_multirun.csv`
and one `_per_class_multirun.csv`):

```
gcn_overall_multirun.csv      gcn_per_class_multirun.csv
gat_overall_multirun.csv      gat_per_class_multirun.csv
gine_overall_multirun.csv     gine_per_class_multirun.csv
gsage_overall_multirun.csv    gsage_per_class_multirun.csv
```

**Output** (4 files):
- `FINAL_overall_ttest_vs_RF.csv` — Accuracy & MCC, GNN vs RF, per architecture
- `FINAL_per_class_ttest_vs_RF.csv` — Precision/Recall/F1 per class, per architecture
- `FINAL_f1_summary_all_architectures.csv` — F1-only view of the above (the "key table")
- `FINAL_table1_overall_meanstd.csv` — human-readable "mean +/- std" summary table

**Method.** Random Forest is deterministic given the fixed train/test split (verified:
identical confusion matrix / classification report across every notebook run), so it is
treated as a fixed reference value (μ₀), not a distribution. Each GNN architecture, in
contrast, was trained 10 times with different fixed seeds, giving a genuine sample of
n=10 independent training outcomes per metric.

A one-sample t-test compares each GNN's 10-run distribution against RF's fixed value:

$$t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}}, \quad df = n - 1$$

where $\bar{x}$ and $s$ are the GNN's across-run mean and standard deviation for that
metric, and $\mu_0$ is RF's fixed value for the same metric. The multirun CSVs store the
*population* standard deviation (NumPy's default, ddof=0); this is converted to the
*sample* standard deviation (ddof=1) required by the t-test before computing $t$ and $p$.

Bonferroni correction (α = 0.05 / 4 = 0.0125) is applied within each metric family
(precision, recall, F1) across the 4 classes, separately per architecture.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

N_RUNS = 10
BONFERRONI_ALPHA = 0.05 / 4  # 4 classes tested per metric family, per architecture

INPUT_DIR = "."   # change if your CSVs live in a different folder
OUTPUT_DIR = "."  # change to write outputs elsewhere

## 2. Random Forest fixed reference values

Taken directly from RF's `classification_report` / confusion matrix (identical across every notebook run, since RF is deterministic given the fixed split).

In [2]:
RF_OVERALL = {
    "Accuracy": 0.7257,
    "MCC": 0.6019,
}

RF_PER_CLASS = {
    "AA": {"precision": 0.7386363636363636, "recall": 0.7558139534883721, "f1": 0.7471264367816092},
    "AI": {"precision": 0.7454545454545455, "recall": 0.6029411764705882, "f1": 0.6666666666666666},
    "IA": {"precision": 0.6666666666666666, "recall": 0.2,                "f1": 0.3076923076923077},
    "II": {"precision": 0.7,                "recall": 0.9032258064516129, "f1": 0.7887323943661971},
}

CLASSES = ["AA", "AI", "IA", "II"]
METRICS_PER_CLASS = ["precision", "recall", "f1"]

# Map architecture name -> input file prefix used by your notebooks
ARCH_FILE_PREFIX = {
    "GCN": "gcn",
    "GAT": "gat",
    "GINE": "gine",
    "GraphSAGE": "gsage",
}

## 3. One-sample t-test helper

In [3]:
def one_sample_ttest_vs_fixed(mu0, xbar, s_pop, n=N_RUNS):
    """
    One-sample t-test of a GNN's n-run distribution (mean=xbar, population
    std=s_pop) against a fixed reference value mu0 (Random Forest).
    Returns (t_statistic, p_value). Returns (nan, nan) if the GNN showed zero
    variance across runs (t-test undefined).
    """
    if s_pop == 0 or np.isnan(s_pop):
        return np.nan, np.nan
    s_sample = s_pop * np.sqrt(n / (n - 1))  # population std (ddof=0) -> sample std (ddof=1)
    se = s_sample / np.sqrt(n)
    t = (xbar - mu0) / se
    df = n - 1
    p = 2 * (1 - stats.t.cdf(abs(t), df))
    return t, p

## 4. Load each architecture's multi-run summary CSVs and run the t-tests

In [4]:
def analyze():
    overall_rows = []
    per_class_rows = []

    for arch, prefix in ARCH_FILE_PREFIX.items():
        overall_df = pd.read_csv(f"{INPUT_DIR}/{prefix}_overall_multirun.csv").set_index("Metric")
        per_class_df = pd.read_csv(f"{INPUT_DIR}/{prefix}_per_class_multirun.csv").set_index("Class")

        # ---- overall: Accuracy, MCC ----
        for metric in ["Accuracy", "MCC"]:
            xbar = overall_df.loc[metric, "Mean"]
            s = overall_df.loc[metric, "Std"]
            mu0 = RF_OVERALL[metric]
            t, p = one_sample_ttest_vs_fixed(mu0, xbar, s)
            overall_rows.append({
                "Architecture": arch, "Metric": metric,
                "RF": mu0, "GNN_mean": xbar, "GNN_std": s,
                "t_stat": t, "p_value": p,
            })

        # ---- per-class: precision, recall, f1 ----
        for cls in CLASSES:
            for metric, col in [("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")]:
                xbar = per_class_df.loc[cls, f"{col}_mean"]
                s = per_class_df.loc[cls, f"{col}_std"]
                mu0 = RF_PER_CLASS[cls][metric]
                t, p = one_sample_ttest_vs_fixed(mu0, xbar, s)
                sig = (p < BONFERRONI_ALPHA) if not np.isnan(p) else False
                per_class_rows.append({
                    "Architecture": arch, "Class": cls, "Metric": metric,
                    "RF": mu0, "GNN_mean": xbar, "GNN_std": s,
                    "diff": xbar - mu0, "t_stat": t, "p_value": p,
                    "significant_bonf": sig,
                })

    overall_result = pd.DataFrame(overall_rows)
    per_class_result = pd.DataFrame(per_class_rows)
    return overall_result, per_class_result


overall_result, per_class_result = analyze()
overall_result.head()

,Architecture,Metric,RF,GNN_mean,GNN_std,t_stat,p_value
0,GCN,Accuracy,0.7257,0.729204,0.014782,0.711057,0.495063
1,GCN,MCC,0.6019,0.606251,0.022909,0.569813,0.582746
2,GAT,Accuracy,0.7257,0.740708,0.010693,4.210606,0.002271
3,GAT,MCC,0.6019,0.626219,0.015336,4.757071,0.001034
4,GINE,Accuracy,0.7257,0.703540,0.017251,-3.853727,0.003884


## 5. Derived tables (F1-only summary, human-readable Table 1)

In [5]:
def build_f1_summary(per_class_result):
    f1_summary = per_class_result[per_class_result["Metric"] == "f1"].copy()
    f1_summary["direction"] = f1_summary["diff"].apply(lambda d: "GNN higher" if d > 0 else "RF higher")
    return f1_summary


def build_table1(overall_result):
    rows = [{"Model": "Random Forest",
             "Accuracy": f"{RF_OVERALL['Accuracy']:.3f} (fixed)",
             "MCC": f"{RF_OVERALL['MCC']:.3f} (fixed)"}]
    for arch in ARCH_FILE_PREFIX:
        sub = overall_result[overall_result.Architecture == arch].set_index("Metric")
        rows.append({
            "Model": arch,
            "Accuracy": f"{sub.loc['Accuracy','GNN_mean']:.3f} +/- {sub.loc['Accuracy','GNN_std']:.3f}",
            "MCC": f"{sub.loc['MCC','GNN_mean']:.3f} +/- {sub.loc['MCC','GNN_std']:.3f}",
        })
    return pd.DataFrame(rows)


f1_summary = build_f1_summary(per_class_result)
table1 = build_table1(overall_result)

## 6. Results

In [6]:
print("=== Table 1: overall performance (mean +/- std) ===")
print(table1.to_string(index=False))

=== Table 1: overall performance (mean +/- std) ===
        Model        Accuracy             MCC
Random Forest   0.726 (fixed)   0.602 (fixed)
          GCN 0.729 +/- 0.015 0.606 +/- 0.023
          GAT 0.741 +/- 0.011 0.626 +/- 0.015
         GINE 0.704 +/- 0.017 0.567 +/- 0.025
    GraphSAGE 0.705 +/- 0.009 0.568 +/- 0.012


In [7]:
print("=== Significant per-class results (Bonferroni-corrected, alpha=0.0125) ===")
sig = per_class_result[per_class_result.significant_bonf]
sig[["Architecture", "Class", "Metric", "RF", "GNN_mean", "diff", "p_value"]]

=== Significant per-class results (Bonferroni-corrected, alpha=0.0125) ===


,Architecture,Class,Metric,RF,GNN_mean,diff,p_value
3,GCN,AI,precision,0.745455,0.696720,-0.048735,5.747480e-06
4,GCN,AI,recall,0.602941,0.741176,0.138235,3.985376e-04
5,GCN,AI,f1,0.666667,0.716133,0.049466,2.759510e-03
6,GCN,IA,precision,0.666667,0.414343,-0.252323,2.119657e-08
7,GCN,IA,recall,0.200000,0.380000,0.180000,4.995768e-05
8,GCN,IA,f1,0.307692,0.395163,0.087471,1.521782e-03
9,GCN,II,precision,0.700000,0.820979,0.120979,1.408714e-04
10,GCN,II,recall,0.903226,0.732258,-0.170968,4.501181e-08
12,GAT,AA,precision,0.738636,0.770213,0.031577,2.987319e-04
13,GAT,AA,recall,0.755814,0.731395,-0.024419,8.226725e-03


## 7. Save outputs

In [8]:
overall_result.to_csv(f"{OUTPUT_DIR}/FINAL_overall_ttest_vs_RF.csv", index=False)
per_class_result.to_csv(f"{OUTPUT_DIR}/FINAL_per_class_ttest_vs_RF.csv", index=False)
f1_summary.to_csv(f"{OUTPUT_DIR}/FINAL_f1_summary_all_architectures.csv", index=False)
table1.to_csv(f"{OUTPUT_DIR}/FINAL_table1_overall_meanstd.csv", index=False)

print("Saved: FINAL_overall_ttest_vs_RF.csv, FINAL_per_class_ttest_vs_RF.csv, "
      "FINAL_f1_summary_all_architectures.csv, FINAL_table1_overall_meanstd.csv")

Saved: FINAL_overall_ttest_vs_RF.csv, FINAL_per_class_ttest_vs_RF.csv, FINAL_f1_summary_all_architectures.csv, FINAL_table1_overall_meanstd.csv
